In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned_pm25.csv")

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset shape: (40983, 15)

Columns:
['date', 'datetime_utc', 'id', 'latitude', 'longitude', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD', 'pm25', 'sensor_name']


In [5]:
df.head()

,date,datetime_utc,id,latitude,longitude,temperature_celsius,pressure_mb,wind_u,wind_v,NO2,CO,O3,AOD,pm25,sensor_name
0,2016-11-25,2016-11-25 00:00:00,1,38.788056,-0.697222,6.493180,946.288660,0.230240,0.735303,NaN,NaN,NaN,0.1110,2.750000,ONTINYENT
1,2016-11-25,2016-11-25 00:00:00,36,39.945278,-0.056389,9.251236,993.626994,0.783218,-0.592618,NaN,NaN,NaN,0.0220,11.666667,ALMASSORA-C. P. OCHANDO
2,2016-11-30,2016-11-30 00:00:00,35,40.051944,-0.189722,10.044782,981.911432,-2.211047,-1.568971,NaN,NaN,NaN,0.2380,2.400000,ALCORA
3,2016-11-30,2016-11-30 00:00:00,37,40.062222,0.072778,12.299584,1008.794766,-3.236886,-2.804465,NaN,NaN,NaN,0.2480,2.600000,BENICASSIM
4,2016-12-02,2016-12-02 00:00:00,1,38.788056,-0.697222,9.350630,953.114180,0.953454,0.293060,NaN,NaN,NaN,0.0755,2.750000,ONTINYENT


In [26]:

# convert timestamp column to datetime
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])

temporal_results = []

# check temporal regularity per station using id
for station_id, group in df.groupby('id'):
    group = group.sort_values('datetime_utc')
    
    time_diffs = group['datetime_utc'].diff().dt.total_seconds()
    median_interval = time_diffs.median()
    std_interval = time_diffs.std()
    
    # compute autocorrelation for pm25 at specific lags
    acf_lag1 = group['pm25'].autocorr(lag=1)
    acf_lag7 = group['pm25'].autocorr(lag=7)
    acf_lag24 = group['pm25'].autocorr(lag=24)
    
    n_days_above_25 = (group['pm25'] > 25).sum()
    n_days_above_50 = (group['pm25'] > 50).sum()
    
    temporal_results.append({
        'station_id': station_id,
        'n_records': len(group),
        't_min': group['datetime_utc'].min(),
        't_max': group['datetime_utc'].max(),
        'sampling_interval_median_sec': median_interval,
        'sampling_interval_std_sec': std_interval,
        'acf_lag1': acf_lag1,
        'acf_lag7': acf_lag7,
        'acf_lag24': acf_lag24,
        'n_days_above_25': n_days_above_25,
        'n_days_above_50': n_days_above_50
    })

df_temporal_summary = pd.DataFrame(temporal_results)

# save the summary to the results directory
df_temporal_summary.to_csv("../results/Exploratory_temporal.csv", index=False)
print(df_temporal_summary.head())

   station_id  n_records      t_min      t_max  sampling_interval_median_sec  \
0           0        727 2016-12-07 2022-03-12                       86400.0   
1           1        447 2016-11-25 2025-09-04                       86400.0   
2           2        498 2024-01-28 2026-02-09                       86400.0   
3           5        726 2016-12-02 2022-02-27                       86400.0   
4           6        717 2016-12-02 2022-03-09                       86400.0   

   sampling_interval_std_sec  acf_lag1  acf_lag7  acf_lag24  n_days_above_25  \
0               6.546300e+05  0.533855  0.121488   0.027186               13   
1               2.210210e+06  0.394208  0.024769   0.024400                4   
2               1.032434e+05  0.488468  0.088889   0.001246                3   
3               6.500043e+05  0.649701  0.198736   0.069400               39   
4               7.148448e+05  0.657281  0.228878   0.200210               44   

   n_days_above_50  
0                